# GRU Representation Manifolds

Analyze the existing best local GRU representation export with descriptive PCA views. Color values are derived from the frozen GRU's phoneme posteriors; they do not influence PCA fitting and require no reference alignment or additional training.

This notebook intentionally loads only the durable `gru` export. Its historical checkpoint provenance is incomplete, so these plots are exploratory repository observations rather than a verified Willett reproduction.

In [ ]:
# Colab / Drive / repo bootstrap.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f'Not running in Colab or Drive already unavailable: {exc}')

import os
import importlib.util
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path('/content/utah-ssl') if Path('/content').exists() else Path.cwd()
REPO_URL = 'https://github.com/ethan-read/utah-ssl.git'

if str(REPO_DIR).startswith('/content'):
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', 'pull', '--ff-only'], cwd=str(REPO_DIR), check=False)

os.chdir(REPO_DIR)
PACKAGE_ROOT = REPO_DIR
os.environ['PYTHONPATH'] = f"{PACKAGE_ROOT}:{os.environ.get('PYTHONPATH', '')}"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT))

required = {'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib', 'sklearn': 'scikit-learn'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)

print('REPO_DIR:', REPO_DIR)
print('PACKAGE_ROOT:', PACKAGE_ROOT)

In [ ]:
from pathlib import Path

# Drive/local roots. Edit these in one place if your Drive layout differs.
DRIVE_ROOT = Path('/content/drive/MyDrive') if Path('/content/drive/MyDrive').exists() else Path('/Users/home/My Drive')
UTAH_SSL_ROOT = DRIVE_ROOT / 'utah_ssl'
REPRESENTATION_ROOT = UTAH_SSL_ROOT / 'data' / 'representations' / 'willett_manifolds'

EXPORT_NAME = 'stanford_released_gru_s5_val_area6v_soft_phonetic_categories_v1'
EXPORT_ROOT = REPRESENTATION_ROOT / EXPORT_NAME
MODEL_KEY = 'gru'
MODEL_DIR = EXPORT_ROOT / MODEL_KEY
TAXONOMY_PATH = REPO_DIR / 'experiments/manifolds/design/articulatory_feature_taxonomy.csv'
PCA_MAX_POINTS = 60_000
PCA_RANDOM_STATE = 7
CONSONANT_POSTERIOR_THRESHOLD = 0.5

MODEL_DIR

In [ ]:
import json
required_export_files = (
    MODEL_DIR / 'metadata.json',
    MODEL_DIR / 'tokens.csv',
    MODEL_DIR / 'examples.csv',
    MODEL_DIR / 'shards.json',
)
missing_export_files = [path for path in required_export_files if not path.exists()]
if missing_export_files:
    raise FileNotFoundError(f'Missing durable GRU export files: {missing_export_files}')
print('Using durable GRU export:', MODEL_DIR)

## Load Exported Artifacts

Everything below uses the saved artifacts, not the model checkpoints.

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path


def load_model_export(export_root, model_key, *, load_input_windows=False):
    model_dir = Path(export_root) / model_key
    metadata = json.loads((model_dir / 'metadata.json').read_text())
    tokens = pd.read_csv(model_dir / 'tokens.csv')
    examples = pd.read_csv(model_dir / 'examples.csv')
    shard_rows = json.loads((model_dir / 'shards.json').read_text())
    hidden_parts = []
    logits_parts = []
    example_index_parts = []
    input_window_parts = []
    adapted_input_window_parts = []
    for shard in shard_rows:
        with np.load(model_dir / 'shards' / shard['shard']) as arrays:
            hidden_parts.append(np.asarray(arrays['hidden']))
            logits_parts.append(np.asarray(arrays['logits']))
            example_index_parts.append(np.asarray(arrays['token_example_index']))
            if load_input_windows:
                if 'input_windows' not in arrays:
                    raise ValueError(f"{model_key}: shard {shard['shard']} lacks required input_windows")
                input_window_parts.append(np.asarray(arrays['input_windows']))
                if 'adapted_input_windows' in arrays:
                    adapted_input_window_parts.append(np.asarray(arrays['adapted_input_windows']))
    hidden = np.concatenate(hidden_parts, axis=0) if hidden_parts else np.zeros((0, 0), dtype=np.float32)
    logits = np.concatenate(logits_parts, axis=0) if logits_parts else np.zeros((0, 0), dtype=np.float32)
    example_indices = np.concatenate(example_index_parts, axis=0) if example_index_parts else np.zeros(0, dtype=np.int64)
    input_windows = np.concatenate(input_window_parts, axis=0) if input_window_parts else None
    adapted_input_windows = np.concatenate(adapted_input_window_parts, axis=0) if adapted_input_window_parts else None
    if not (hidden.shape[0] == logits.shape[0] == example_indices.shape[0] == tokens.shape[0] == int(metadata['token_count'])):
        raise ValueError(f'{model_key}: exported token counts disagree')
    if hidden.shape[1] != int(metadata['hidden_dim']) or logits.shape[1] != int(metadata['vocab']['num_classes']):
        raise ValueError(f'{model_key}: hidden/logit dimensions disagree with metadata')
    if not np.array_equal(example_indices, tokens['example_export_index'].to_numpy(dtype=np.int64)):
        raise ValueError(f'{model_key}: shard example indices do not align with tokens.csv')
    if input_windows is not None and input_windows.shape[0] != tokens.shape[0]:
        raise ValueError(f'{model_key}: input-window rows {input_windows.shape[0]} != token rows {tokens.shape[0]}')
    if adapted_input_windows is not None and adapted_input_windows.shape[0] != tokens.shape[0]:
        raise ValueError(f'{model_key}: adapted-input-window rows {adapted_input_windows.shape[0]} != token rows {tokens.shape[0]}')
    return {
        'metadata': metadata,
        'tokens': tokens,
        'examples': examples,
        'hidden': hidden,
        'logits': logits,
        'input_windows': input_windows,
        'adapted_input_windows': adapted_input_windows,
    }


payload = load_model_export(EXPORT_ROOT, MODEL_KEY, load_input_windows=False)
exports = {MODEL_KEY: payload}
meta = payload['metadata']
display(pd.DataFrame([{
    'model': MODEL_KEY,
    'model_dir': str(MODEL_DIR),
    'dataset': meta.get('dataset'),
    'examples': meta['example_count'],
    'tokens': meta['token_count'],
    'hidden_dim': meta['hidden_dim'],
    'input_window_dim': meta.get('input_window_dim'),
    'checkpoint_step': meta['checkpoint_step'],
    'patch_ms': meta['patch_size_ms'],
    'stride_ms': meta['patch_stride_ms'],
}]))


## PCA Views

These are descriptive projections of the frozen GRU hidden states. The broad and articulator colors are posterior sums derived from the existing phoneme head; they do not influence PCA fitting. Interpret color gradients more than apparent clusters.

In [ ]:
import matplotlib.pyplot as plt
from experiments.manifolds.representation_export import id_to_symbol_from_vocab
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


def fit_pca_view(hidden, max_points=60_000, seed=7):
    rng = np.random.default_rng(seed)
    n = hidden.shape[0]
    if n > max_points:
        idx = np.sort(rng.choice(n, size=max_points, replace=False))
    else:
        idx = np.arange(n)
    x = hidden[idx].astype(np.float32, copy=False)
    x = StandardScaler().fit_transform(x)
    pca = PCA(n_components=3, random_state=seed)
    pcs = pca.fit_transform(x)
    return idx, pcs, pca

ARTICULATOR_TARGETS = ('lips', 'tongue_front', 'tongue_body')
taxonomy_frame = pd.read_csv(TAXONOMY_PATH, keep_default_na=False)
export_id_to_symbol = id_to_symbol_from_vocab(payload['metadata']['vocab'])
taxonomy_id_to_symbol = dict(zip(taxonomy_frame['phoneme_id'].astype(int), taxonomy_frame['symbol'].astype(str)))
if export_id_to_symbol != taxonomy_id_to_symbol:
    raise ValueError('Canonical taxonomy symbols do not match the exported GRU vocabulary.')


def articulator_posterior_frame(logits, taxonomy, targets=ARTICULATOR_TARGETS):
    """Sum frozen phoneme probabilities over canonical consonant articulators."""
    logits = np.asarray(logits, dtype=np.float64)
    shifted = logits - logits.max(axis=1, keepdims=True)
    probabilities = np.exp(shifted)
    probabilities /= probabilities.sum(axis=1, keepdims=True)
    membership = np.zeros((logits.shape[1], len(targets)), dtype=np.float64)
    consonant_membership = np.zeros(logits.shape[1], dtype=np.float64)
    for row in taxonomy.itertuples(index=False):
        phoneme_id = int(row.phoneme_id)
        if phoneme_id < 0 or phoneme_id >= logits.shape[1]:
            raise ValueError(f'Taxonomy phoneme ID {phoneme_id} is outside the logit vocabulary.')
        if row.segment_family != 'consonant':
            continue
        consonant_membership[phoneme_id] = 1.0
        articulators = set(str(row.primary_articulators).split('|'))
        for target_index, target in enumerate(targets):
            membership[phoneme_id, target_index] = float(target in articulators)
    if set(taxonomy['phoneme_id'].astype(int)) != set(range(logits.shape[1])):
        raise ValueError('Taxonomy IDs do not exactly match the GRU logit vocabulary.')
    articulator_probabilities = probabilities @ membership
    consonant_probability = probabilities @ consonant_membership
    result = pd.DataFrame({
        f'{target}_prob': articulator_probabilities[:, target_index]
        for target_index, target in enumerate(targets)
    })
    result['derived_consonant_prob'] = consonant_probability
    for target in targets:
        result[f'{target}_given_consonant'] = (
            result[f'{target}_prob'] / np.maximum(consonant_probability, 1e-12)
        )
    probability_columns = [column for column in result if column != 'derived_consonant_prob']
    if not np.isfinite(result.to_numpy()).all():
        raise ValueError('Derived articulator posteriors contain nonfinite values.')
    if not ((result[probability_columns] >= -1e-8) & (result[probability_columns] <= 1 + 1e-8)).all().all():
        raise ValueError('Derived articulator posteriors fall outside [0, 1].')
    return result


def plot_pca_colors(frame, columns, *, title, mask=None):
    selected = frame if mask is None else frame.loc[np.asarray(mask, dtype=bool)]
    if selected.empty:
        raise ValueError(f'No PCA points remain for {title!r}.')
    fig, axes = plt.subplots(1, len(columns), figsize=(4.2 * len(columns), 3.8), sharex=True, sharey=True)
    axes = np.atleast_1d(axes)
    for ax, column in zip(axes, columns):
        sc = ax.scatter(selected['pc1'], selected['pc2'], c=selected[column], s=3, cmap='viridis', vmin=0 if column != 'entropy_bits' else None, vmax=1 if column != 'entropy_bits' else None, alpha=0.65, linewidths=0)
        ax.set_title(column)
        ax.set_xlabel('PC1')
        ax.set_ylabel('PC2')
        fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
    fig.suptitle(title, y=1.03)
    fig.tight_layout()
    plt.show()


idx, pcs, pca = fit_pca_view(payload['hidden'], max_points=PCA_MAX_POINTS, seed=PCA_RANDOM_STATE)
frame = payload['tokens'].iloc[idx].reset_index(drop=True).copy()
frame[['pc1', 'pc2', 'pc3']] = pcs
articulator_posteriors = articulator_posterior_frame(payload['logits'][idx], taxonomy_frame)
frame = pd.concat([frame, articulator_posteriors], axis=1)
if not np.allclose(frame['derived_consonant_prob'], frame['consonant_prob'], atol=2e-5):
    raise ValueError('Taxonomy-derived consonant probability does not match the exported consonant probability.')
pca_payloads = {MODEL_KEY: {'frame': frame, 'pca': pca, 'indices': idx}}
print(MODEL_KEY, 'explained variance:', np.round(pca.explained_variance_ratio_, 4))

plot_pca_colors(
    frame,
    ['vowel_prob', 'consonant_prob', 'blank_prob', 'silence_prob', 'entropy_bits'],
    title=f'{MODEL_KEY}: broad frozen-GRU posteriors',
)
plot_pca_colors(
    frame,
    [f'{target}_prob' for target in ARTICULATOR_TARGETS],
    title=f'{MODEL_KEY}: unconditional consonant-articulator posteriors',
)
high_consonant = frame['derived_consonant_prob'] >= CONSONANT_POSTERIOR_THRESHOLD
print(f'High-consonant points: {int(high_consonant.sum()):,} / {len(frame):,}')
plot_pca_colors(
    frame,
    [f'{target}_given_consonant' for target in ARTICULATOR_TARGETS],
    title=f'{MODEL_KEY}: articulator posteriors conditional on P(consonant) >= {CONSONANT_POSTERIOR_THRESHOLD:g}',
    mask=high_consonant,
)

## Matched Raw-Input PCA Baseline

This is the direct control for the hidden-state PCA above: it fits PCA to the raw neural input windows for the same frames, then colors those points using the GRU's logits-derived probabilities. It therefore tests whether a structure such as the vowel zone was already visible in the neural inputs before the GRU transformed them.


In [ ]:
RAW_BASELINE_MODEL_KEY = MODEL_KEY
RAW_BASELINE_WINDOW_KEY = 'input_windows'  # Set to 'adapted_input_windows' for the post-adapter control.
RAW_BASELINE_MAX_POINTS = 60_000
RAW_BASELINE_RANDOM_STATE = 7

if exports[RAW_BASELINE_MODEL_KEY].get(RAW_BASELINE_WINDOW_KEY) is None:
    exports[RAW_BASELINE_MODEL_KEY] = load_model_export(
        EXPORT_ROOT,
        RAW_BASELINE_MODEL_KEY,
        load_input_windows=True,
    )

# Reuse the hidden-state PCA sample whenever it exists, so the two panels differ
# only in the representation being projected. Fall back to the same seeded sampling rule.
hidden_pca_sample = pca_payloads.get(RAW_BASELINE_MODEL_KEY, {}).get('indices')
n_tokens = len(exports[RAW_BASELINE_MODEL_KEY]['tokens'])
if hidden_pca_sample is not None:
    baseline_indices = np.asarray(hidden_pca_sample, dtype=np.int64)
    sample_source = 'reused hidden-state PCA sample'
else:
    rng = np.random.default_rng(RAW_BASELINE_RANDOM_STATE)
    baseline_indices = (
        np.sort(rng.choice(n_tokens, size=RAW_BASELINE_MAX_POINTS, replace=False))
        if n_tokens > RAW_BASELINE_MAX_POINTS
        else np.arange(n_tokens)
    )
    sample_source = 'seeded all-frame sample'

baseline_windows = exports[RAW_BASELINE_MODEL_KEY][RAW_BASELINE_WINDOW_KEY][baseline_indices].astype(np.float32, copy=False)
baseline_pca = PCA(n_components=3, svd_solver='randomized', random_state=RAW_BASELINE_RANDOM_STATE)
baseline_pcs = baseline_pca.fit_transform(StandardScaler().fit_transform(baseline_windows))
raw_baseline_frame = exports[RAW_BASELINE_MODEL_KEY]['tokens'].iloc[baseline_indices].reset_index(drop=True).copy()
raw_baseline_frame['pc1'] = baseline_pcs[:, 0]
raw_baseline_frame['pc2'] = baseline_pcs[:, 1]
raw_baseline_frame['pc3'] = baseline_pcs[:, 2]

print(
    f'{RAW_BASELINE_MODEL_KEY}: {RAW_BASELINE_WINDOW_KEY}; {len(raw_baseline_frame):,} frames; {sample_source}; '
    f'explained variance {np.round(baseline_pca.explained_variance_ratio_, 4)}'
)

baseline_color_columns = ['vowel_prob', 'consonant_prob', 'blank_prob', 'silence_prob', 'entropy_bits']
fig, axes = plt.subplots(1, len(baseline_color_columns), figsize=(4.2 * len(baseline_color_columns), 3.8), sharex=True, sharey=True)
for ax, column in zip(axes, baseline_color_columns):
    sc = ax.scatter(
        raw_baseline_frame['pc1'], raw_baseline_frame['pc2'],
        c=raw_baseline_frame[column], s=3, cmap='viridis', alpha=0.65, linewidths=0,
    )
    ax.set_title(f'{RAW_BASELINE_MODEL_KEY}: {column}')
    ax.set_xlabel('raw-input PC1')
    ax.set_ylabel('raw-input PC2')
    fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
plt.suptitle(f'{RAW_BASELINE_MODEL_KEY}: raw neural input PCA colored by GRU beliefs', y=1.03)
plt.tight_layout()
plt.show()


## Save Analysis Tables

Optional compact outputs for later plotting outside the notebook.

In [ ]:
ANALYSIS_DIR = EXPORT_ROOT / 'analysis_tables'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

summary = payload['tokens'].groupby('top_category').agg(
    token_count=('global_token_index', 'count'),
    mean_top1_prob=('top1_prob', 'mean'),
    mean_entropy_bits=('entropy_bits', 'mean'),
    mean_vowel_prob=('vowel_prob', 'mean'),
    mean_consonant_prob=('consonant_prob', 'mean'),
).reset_index()
summary.to_csv(ANALYSIS_DIR / f'{MODEL_KEY}_category_summary.csv', index=False)
pca_payloads[MODEL_KEY]['frame'].to_csv(ANALYSIS_DIR / f'{MODEL_KEY}_pca_sample.csv', index=False)

print('Saved analysis tables to:', ANALYSIS_DIR)

In [ ]:
from google.colab import drive, runtime
drive.flush_and_unmount()
runtime.unassign()